# Copy Number Variation (CNV) Analysis using SeSAMe & Conumee2

## 📌 Installation & Loading Packages
```r
if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager")
BiocManager::install(c("sesame", "conumee2", "TCGAbiolinks", "GEOquery",
                       "IlluminaHumanMethylation450kanno.ilmn12.hg19",
                       "IlluminaHumanMethylationEPICanno.ilm10b4.hg19",
                       "BiocParallel", "dplyr"))
remove.packages("conumee2.0")
devtools::install_github("hovestadtlab/conumee2", subdir = "conumee2")
```

## 📌 Load Required Libraries
```r
library(sesame)        # Methylation processing
library(conumee2)      # CNV analysis
library(dplyr)         # Data manipulation
library(BiocParallel)  # Parallel processing
library(TCGAbiolinks)  # GDC Data Querying
library(GEOquery)      # GEO Data Querying
register(MulticoreParam(7))  # Use 7 CPU cores
```

## 📂 Define Paths
```r
### 📂 Define Paths
idatPath_cases <- "/Users/nijat/Downloads/Glioma_CNV_Data_2025/methylation/idat_file"
idatPath_controls <- "/Users/nijat/Downloads/Glioma_CNV_Data_2025/methylation/control_files"
analysisPath <- "/Users/nijat/Downloads/Glioma_CNV_Data_2025/methylation/CNV_result"

sample_sheet_cases <- "/Users/nijat/Downloads/Glioma_CNV_Data_2025/methylation/Sample_Sheet.csv"
sample_sheet_controls <- "/Users/nijat/Downloads/Glioma_CNV_Data_2025/methylation/Control_Sample_Sheet.csv"

if (!dir.exists(analysisPath)) dir.create(analysisPath, recursive = TRUE)
```

## 📌 Load and Clean Sample Sheets
```r
clean_sample_sheet <- function(sample_sheet) {
  targets <- read.csv(sample_sheet, stringsAsFactors = FALSE)
  if (!all(c("Sentrix_ID", "Sentrix_Position", "Sample_Name") %in% colnames(targets))) {
    stop("Missing required columns!")
  }
  targets <- targets %>% distinct() %>% group_by(Sample_Name) %>% slice(1) %>% ungroup()
  return(targets)
}
targets_cases <- clean_sample_sheet(sample_sheet_cases)
targets_controls <- clean_sample_sheet(sample_sheet_controls)
```

## 📌 Read IDAT Files Using SeSAMe
```r
sdfs.q <- openSesame(idatPath_cases, prep = "QCDPB", BPPARAM = MulticoreParam(7))
sdfs.c <- openSesame(idatPath_controls, prep = "QCDPB", BPPARAM = MulticoreParam(7))
data.q <- CNV.load(do.call(cbind, lapply(sdfs.q, totalIntensities)))
data.c <- CNV.load(do.call(cbind, lapply(sdfs.c, totalIntensities)))
```

## 📌 Define CNV Annotations
```r
data(exclude_regions)
data(detail_regions)
anno <- CNV.create_anno(array_type = c("450k", "EPICv2"), exclude_regions = exclude_regions, detail_regions = detail_regions)
```

## 📌 CNV Analysis Pipeline
```r
cnv_result <- CNV.fit(data.q, data.c, anno)
cnv_result <- CNV.bin(cnv_result)
cnv_result <- CNV.detail(cnv_result)
cnv_result <- CNV.segment(cnv_result)
cnv_result <- CNV.focal(cnv_result)
saveRDS(cnv_result, file = file.path(analysisPath, "cnv_result.rds"))
```

## 📊 Generate CNV Plots
```r
pdf(file.path(analysisPath, "cnv_genomeplot300.pdf"))
CNV.genomeplot(cnv_result[300])
dev.off()
```

## 📌 Save CNV Results
```r
segments <- CNV.write(cnv_result, what = "segments")
write.csv(segments, file = file.path(analysisPath, "CNV_segments_results.csv"), row.names = FALSE)
```

## 📊 Interactive Visualization
```r
if (!requireNamespace("plotly", quietly = TRUE)) install.packages("plotly")
library(plotly)
interactive_plot <- CNV.plotly(cnv_result[3])
htmlwidgets::saveWidget(interactive_plot, file = file.path(analysisPath, "cnv_genomeplot_interactive.html"))
```

## 📌 Extract & Filter CNV Segments
```r
cnv_segments_list <- CNV.write(cnv_result, what = "segments")
cnv_segments <- if (is.list(cnv_segments_list)) bind_rows(cnv_segments_list, .id = "sampleID") else as.data.frame(cnv_segments_list)
significant_cnv <- cnv_segments %>% filter(seg.mean > 0.5 | seg.mean < -0.5, pval < 0.05, num.mark >= 15)
```

## 📊 CNV Visualization
```r
library(ggplot2)
ggplot(significant_cnv, aes(x = chrom, y = seg.mean, color = sampleID)) +
  geom_point(size = 3) + theme_minimal() + labs(title = "Significant CNVs", x = "Chromosome", y = "Log2 Copy Number Ratio") +
  theme(axis.text.x = element_text(angle = 90, hjust = 1))
```

## 📌 Save Additional CNV Data
```r
saveRDS(CNV.write(cnv_result, what = "bins"), file = file.path(analysisPath, "cnv_bins.rds"))
saveRDS(CNV.write(cnv_result, what = "detail"), file = file.path(analysisPath, "cnv_detail.rds"))
saveRDS(CNV.write(cnv_result, what = "probes"), file = file.path(analysisPath, "cnv_probes.rds"))
saveRDS(CNV.write(cnv_result, what = "focal"), file = file.path(analysisPath, "cnv_focal.rds"))
